# 08 · SAINT Baseline (Lending Club)

Reference implementation of the **SAINT** architecture (Self-Attention and Intersample Attention Transformer) trained on the Lending Club default-risk dataset. SAINT extends a transformer-style tabular model with intersample attention, allowing it to share information across samples in the same batch.

This notebook serves as the SAINT reference point for comparison with the FT-Transformer experiments in `02_…` through `06_…`.

## 1. Setup

In [ ]:
# %pip install rtdl_revisiting_models -q
%pip install delu -q
%pip install optuna -q

In [ ]:
import math
import random
import warnings
import json
import os
import itertools
import typing
from collections import OrderedDict
from pathlib import Path
from typing import Any, Dict, Iterable, List, Literal, Optional, Tuple, cast

import numpy as np
import pandas as pd
import scipy.special

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim
from torch import Tensor
from torch.nn.parameter import Parameter
from torch.utils.data import Dataset, DataLoader, TensorDataset

import sklearn.datasets
import sklearn.metrics
import sklearn.model_selection
import sklearn.preprocessing
import sklearn.tree as sklearn_tree
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder, QuantileTransformer
from sklearn.metrics import roc_auc_score
from sklearn.isotonic import IsotonicRegression

import optuna
import delu
from tqdm.std import tqdm
from tqdm import tqdm as _tqdm  # alias for cells that use it directly

warnings.filterwarnings("ignore")
pd.options.display.max_rows = 200
pd.options.display.max_columns = 200

RANDOM_SEED = 42


def seed_everything(seed: int = RANDOM_SEED) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


seed_everything()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

In [ ]:
# --- Optional: Google Colab Drive mount ---
# Uncomment the three lines below if you're running on Colab and want to read
# data from / write artifacts to your Drive. Skip on local, server, or Kaggle
# runs. The next cell auto-routes through DRIVE_ROOT when defined.
#
# from google.colab import drive
# drive.mount("/content/drive")
# DRIVE_ROOT = "/content/drive/MyDrive/ft-transformer-credit-risk-study"

# When running locally, repo root is one directory up (notebook is in
# notebooks/<dataset>/). On Colab with the cell above uncommented, DRIVE_ROOT
# takes precedence.
_BASE = globals().get("DRIVE_ROOT", "..")
DATA_PATH      = f"{_BASE}/data/processed_data_densest.parquet.gzip"
ARTIFACTS_DIR  = Path(f"{_BASE}/artifacts/lending_club")
RESULTS_DIR    = ARTIFACTS_DIR / "results"
MODELS_DIR     = ARTIFACTS_DIR / "models"
FIGURES_DIR    = ARTIFACTS_DIR / "figures"
for _d in (ARTIFACTS_DIR, RESULTS_DIR, MODELS_DIR, FIGURES_DIR):
    _d.mkdir(parents=True, exist_ok=True)

print(f"DATA_PATH      = {DATA_PATH}")
print(f"ARTIFACTS_DIR  = {ARTIFACTS_DIR}")

In [ ]:
# ──────────────────────────────────────────────────────────────────────────────
# DEV_MODE — smoke-test switch.
#
# When False (the default), every constant resolves to the exact value used in
# the original experiment. The DEV_MODE branches below are dead code in that
# path, so the run is behaviourally identical to the source notebook.
#
# When True, the notebook runs a tiny fast version: a subsample of the data,
# a couple of epochs, a single Optuna trial. Use this on Colab to validate the
# full pipeline end-to-end before launching a real run.
# ──────────────────────────────────────────────────────────────────────────────
DEV_MODE = False
if DEV_MODE:
    print("DEV_MODE is ON — smoke-test run with reduced data/epochs/trials.")

## 2. Data Loading

In [ ]:
# Load preprocessed Lending Club dataset.
df4 = pd.read_parquet(DATA_PATH)

train_val_df, test_df = train_test_split(
    df4, test_size=0.2,
    stratify=df4["target_binary"], random_state=RANDOM_SEED,
)
train_df, val_df = train_test_split(
    train_val_df, test_size=0.2,
    stratify=train_val_df["target_binary"], random_state=RANDOM_SEED,
)
print("Train_val / test sizes:", len(train_val_df), len(test_df))
print("Train / val sizes:", len(train_df), len(val_df))

cat_columns = [
    "term", "emp_length", "home_ownership", "verification_status",
    "purpose", "zip_code", "addr_state", "application_type",
    "initial_list_status", "disbursement_method",
]
num_cols = [
    c for c in list(df4.columns)
    if c not in cat_columns + ["id", "emp_title", "target_binary"]
]
print(f"# numerical features: {len(num_cols)}  |  # categorical: {len(cat_columns)}")

## 3. Preprocessing

Median imputation + `StandardScaler` for numerical features, label encoding for categorical features with an extra `UNKNOWN` bucket so that unseen categories in validation/test do not break the model.

In [ ]:
# Numeric imputation + scaling (median + StandardScaler).
num_impute = {}
scaler = StandardScaler()

num_df = train_df[num_cols].copy()
for c in num_df.columns:
    if num_df[c].dtype == "object":
        num_df[c] = pd.to_numeric(num_df[c], errors="coerce")
    med = num_df[c].median()
    num_df[c] = num_df[c].fillna(med)
    num_impute[c] = med
num_df[num_df.columns] = scaler.fit_transform(num_df[num_df.columns])

# Label encoding with a permanent UNKNOWN bucket for unseen validation/test categories.
cat_encoders: Dict[str, LabelEncoder] = {}
cat_cardinalities = []
cat_df = train_df[cat_columns].copy()
for c in cat_columns:
    le = LabelEncoder()
    series = cat_df[c].fillna("MISSING").astype(str)
    le.fit(series)
    new_classes = list(le.classes_)
    if "MISSING" not in new_classes:
        new_classes.append("MISSING")
    new_classes.append("UNKNOWN")
    le.classes_ = np.array(new_classes)
    cat_df[c + "_le"] = le.transform(series)
    cat_encoders[c] = le
    cat_cardinalities.append(len(le.classes_))

train_df_proc = pd.concat(
    [
        num_df.reset_index(drop=True),
        cat_df[[c + "_le" for c in cat_columns]].reset_index(drop=True),
        train_df["target_binary"].reset_index(drop=True),
    ],
    axis=1,
)
print("Encoded train")

# Apply the same imputation, scaling and label maps to val and test.
def _apply_transforms(part_df):
    num_part = part_df[num_cols].copy()
    for c in num_part.columns:
        if num_part[c].dtype == "object":
            num_part[c] = pd.to_numeric(num_part[c], errors="coerce")
    for c in num_part.columns:
        num_part[c] = num_part[c].fillna(num_impute[c])
    num_part = pd.DataFrame(
        scaler.transform(num_part), columns=num_part.columns, index=num_part.index
    )

    cat_part = part_df[cat_columns].copy()
    for c in cat_columns:
        le = cat_encoders[c]
        mapping = {cls: i for i, cls in enumerate(le.classes_)}
        unknown_id = mapping["UNKNOWN"]
        series = cat_part[c].fillna("MISSING").astype(str)
        cat_part[c + "_le"] = series.map(mapping).fillna(unknown_id).astype(int)

    return pd.concat(
        [
            num_part.reset_index(drop=True),
            cat_part[[c + "_le" for c in cat_columns]].reset_index(drop=True),
            part_df["target_binary"].reset_index(drop=True),
        ],
        axis=1,
    )


val_df_proc = _apply_transforms(val_df)
test_df_proc = _apply_transforms(test_df)
print("Encoded val and test")

In [ ]:
# Materialize everything as torch tensors on the target device.
cat_le_cols = [c + "_le" for c in cat_columns]
data_numpy = {"train": {}, "val": {}, "test": {}}
for part_name, part_df in [
    ("train", train_df_proc),
    ("val",   val_df_proc),
    ("test",  test_df_proc),
]:
    data_numpy[part_name]["X_cont"] = part_df[num_cols]
    data_numpy[part_name]["x_cat"]  = part_df[cat_le_cols]
    data_numpy[part_name]["y"]      = part_df["target_binary"]

data = {}
for part in data_numpy:
    data[part] = {
        "X_cont": torch.tensor(data_numpy[part]["X_cont"].to_numpy(), dtype=torch.float32, device=device),
        "x_cat":  torch.tensor(data_numpy[part]["x_cat"].to_numpy(),  dtype=torch.long,    device=device),
        "y":      torch.tensor(data_numpy[part]["y"].to_numpy(),      dtype=torch.float32, device=device),
    }

d_numerical = len(num_cols)
n_train = data["train"]["y"].shape[0]
print(f"d_numerical = {d_numerical}")
print(f"cat_cardinalities = {cat_cardinalities}")
print(f"# train rows = {n_train}")

In [ ]:
# When DEV_MODE is on, subsample data to a few thousand rows so a full training
# pass completes in a couple of minutes. No-op when DEV_MODE is False.
if DEV_MODE:
    _n_dev = 5000
    for _part in data:
        _idx = torch.randperm(data[_part]["y"].shape[0], device=device)[:_n_dev]
        for _k in data[_part]:
            data[_part][_k] = data[_part][_k][_idx]
    print({_part: {k: v.shape for k, v in data[_part].items()} for _part in data})

## 4. Model Definition

Light-weight SAINT implementation: self-attention over feature tokens followed by per-row layer norm and a feed-forward block, repeated `depth` times. The final classification head concatenates all token embeddings and projects to a single logit.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class SaintAttention(nn.Module):
    def __init__(self, dim, heads=8, dropout=0.1):
        super().__init__()
        self.heads = heads
        self.scale = (dim // heads) ** -0.5
        self.to_qkv = nn.Linear(dim, dim * 3, bias=False)
        self.to_out = nn.Linear(dim, dim)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        b, n, d = x.shape
        qkv = self.to_qkv(x).chunk(3, dim=-1)
        q, k, v = map(lambda t: t.view(b, n, self.heads, -1).transpose(1, 2), qkv)

        dots = torch.matmul(q, k.transpose(-1, -2)) * self.scale
        attn = F.softmax(dots, dim=-1)
        out = torch.matmul(self.dropout(attn), v)
        out = out.transpose(1, 2).reshape(b, n, d)
        return self.to_out(out)

class SAINT(nn.Module):
    def __init__(self, categories, num_continuous, dim=32, depth=6, heads=8):
        super().__init__()
        self.num_categories = len(categories)
        self.num_continuous = num_continuous

        # Embeddings
        self.cat_embeds = nn.ModuleList([nn.Embedding(c, dim) for c in categories])
        self.cont_embed = nn.Linear(1, dim)

        # Transformer Layers
        self.layers = nn.ModuleList([])
        for _ in range(depth):
            self.layers.append(nn.ModuleList([
                SaintAttention(dim, heads), # Column Attention
                nn.LayerNorm(dim),
                nn.Linear(dim, dim), # Feed Forward
                nn.LayerNorm(dim)
            ]))

        self.mlp_head = nn.Sequential(
            nn.LayerNorm(dim * (self.num_categories + num_continuous)),
            nn.Linear(dim * (self.num_categories + num_continuous), 1)
        )

    def forward(self, x_cat, x_cont):
        # x_cat: [Batch, n_cat], x_cont: [Batch, n_cont]
        embeddings = []
        for i, emb in enumerate(self.cat_embeds):
            embeddings.append(emb(x_cat[:, i]))

        for i in range(self.num_continuous):
            embeddings.append(self.cont_embed(x_cont[:, i:i+1]))

        x = torch.stack(embeddings, dim=1) # [Batch, n_features, dim]

        for attn, norm1, ff, norm2 in self.layers:
            x = attn(x) + x
            x = norm1(x)
            x = ff(x) + x
            x = norm2(x)

        return self.mlp_head(x.flatten(1))

In [ ]:
from torch.utils.data import DataLoader, TensorDataset

from torch.utils.data import DataLoader, TensorDataset

# Define batch size here - remember, larger batches give SAINT more context!
BATCH_SIZE = 1024

train_ds = TensorDataset(data["train"]["x_cat"], data["train"]["X_cont"], data["train"]["y"])
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)

val_ds = TensorDataset(data["val"]["x_cat"], data["val"]["X_cont"], data["val"]["y"])
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False)

test_ds = TensorDataset(data["test"]["x_cat"], data["test"]["X_cont"], data["test"]["y"])
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False)


print(f"Created train_loader with {len(train_loader)} batches")
print(f"Created val_loader with {len(val_loader)} batches")
print(f"Created val_loader with {len(test_loader)} batches")

## 5. Optuna Optimization

The search space covers `dim ∈ {32, 64, 128, 256}`, `depth ∈ [2, 8]`, `heads ∈ {2, 4, 8}` and weight decay. Best trial's checkpoint is written to `MODELS_DIR / 'saint_lc_best.pth'` and the full set of trial results to `RESULTS_DIR / 'saint_lc_trials.json'`.

In [ ]:
import json
import math
import optuna
import torch
import torch.nn as nn
import torch.nn.functional as F
from tqdm import tqdm
from sklearn.metrics import roc_auc_score

# ─────────────────────────────────────────────
# GLOBALS
# ─────────────────────────────────────────────
trial_results = []
global_best_auc = 0.0
global_model_path = str(MODELS_DIR / "saint_lc_best.pth")

# ─────────────────────────────────────────────
# EVALUATE HELPER
# ─────────────────────────────────────────────
@torch.no_grad()
def evaluate(model, loader, device):
    model.eval()
    all_preds, all_targets = [], []
    for batch_cat, batch_cont, batch_y in loader:
        logits = model(batch_cat.to(device), batch_cont.to(device)).squeeze(-1)
        probs = torch.sigmoid(logits)
        all_preds.extend(probs.cpu().numpy())
        all_targets.extend(batch_y.cpu().numpy())
    return roc_auc_score(all_targets, all_preds)

# ─────────────────────────────────────────────
# OBJECTIVE
# ─────────────────────────────────────────────
def objective(trial):
    global global_best_auc

    # ── Search Space ─────────────────────────
    dim = trial.suggest_categorical("dim", [32, 64, 128, 256])
    depth = trial.suggest_int("depth", 2, 8)
    heads = trial.suggest_categorical("heads", [2, 4, 8])
    # lr = trial.suggest_float("lr", 1e-5, 1e-3, log=True)
    lr = 1e-4
    weight_decay = trial.suggest_float("weight_decay", 1e-6, 1e-2, log=True)
    # n_epochs = trial.suggest_int("n_epochs", 5, 30)
    n_epochs = 30
    patience = 10

    # dim must be divisible by heads
    if dim % heads != 0:
        raise optuna.exceptions.TrialPruned()

    # ── Print Trial Info ──────────────────────
    print("\n" + "=" * 70)
    print(f" Starting Trial {trial.number}")
    print(" Hyperparameters:")
    print(f" dim = {dim}")
    print(f" depth = {depth}")
    print(f" heads = {heads}")
    # print(f" lr = {lr:.2e}")
    print(f" weight_decay = {weight_decay:.2e}")
    # print(f" n_epochs = {n_epochs}")
    print("=" * 70 + "\n")

    # ── Model ─────────────────────────────────
    model = SAINT(
        categories=cat_cardinalities,
        num_continuous=d_numerical,
        dim=dim,
        depth=depth,
        heads=heads,
    ).to(device)

    optimizer = torch.optim.AdamW(
        model.parameters(), lr=lr, weight_decay=weight_decay
    )
    scheduler = torch.optim.lr_scheduler.OneCycleLR(
        optimizer,
        max_lr=lr * 5,
        steps_per_epoch=len(train_loader),
        epochs=n_epochs,
        pct_start=0.3,
        anneal_strategy="cos",
    )
    criterion = nn.BCEWithLogitsLoss()

    # ── Early Stopping State ──────────────────
    best = {"val": -math.inf, "test": -math.inf, "epoch": -1}
    epochs_no_improve = 0

    print(f"Score before training: Val AUC = {evaluate(model, val_loader, device):.4f}")
    print(f"Device: {device}")
    print("-" * 70)

    # ── Training Loop ─────────────────────────
    for epoch in range(n_epochs):
        model.train()
        train_loss = 0.0

        pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{n_epochs} [Train]")
        for batch_cat, batch_cont, batch_y in pbar:
            optimizer.zero_grad()
            logits = model(
                batch_cat.to(device), batch_cont.to(device)
            ).squeeze(-1)
            loss = criterion(logits, batch_y.to(device).float())
            loss.backward()
            optimizer.step()
            scheduler.step()
            train_loss += loss.item()
            pbar.set_postfix(loss=loss.item())

        # ── Validation ────────────────────────
        val_auc = evaluate(model, val_loader, device)
        test_auc = evaluate(model, test_loader, device)
        print(f" AUC Val {val_auc:.4f} Test {test_auc:.4f}")

        # Optuna pruning
        trial.report(val_auc, epoch)
        if trial.should_prune():
            raise optuna.exceptions.TrialPruned()

        # Early stopping + best tracking
        if val_auc > best["val"]:
            print(" New best epoch! ")
            best = {"val": val_auc, "test": test_auc, "epoch": epoch + 1}
            epochs_no_improve = 0
            if val_auc > global_best_auc:
                torch.save(model.state_dict(), global_model_path)
                global_best_auc = val_auc
        else:
            epochs_no_improve += 1
            if epochs_no_improve >= patience:
                print(f"⏹ Early stopping at epoch {epoch+1}")
                break

    # ── Save Trial Result ─────────────────────
    trial_results.append({
        "trial_number": trial.number,
        "dim": dim,
        "depth": depth,
        "heads": heads,
        # "lr": lr,
        "weight_decay": weight_decay,
        # "n_epochs": n_epochs,
        "best": best,
        "auc_val": val_auc,
        "auc_test": test_auc,
    })
    with open(str(RESULTS_DIR / "saint_lc_trials.json"), "w") as f:
        json.dump(trial_results, f, indent=4)

    print(f"\nResult: {best}")
    return best["val"]


# ─────────────────────────────────────────────
# RUN STUDY
# ─────────────────────────────────────────────
sampler = optuna.samplers.TPESampler(seed=42)
pruner = optuna.pruners.MedianPruner(n_startup_trials=5, n_warmup_steps=3)

study = optuna.create_study(
    direction="maximize",
    sampler=sampler,
    pruner=pruner,
    study_name="saint_hyperparam_search",
)
study.optimize(objective, n_trials=(1 if DEV_MODE else 30))

# ─────────────────────────────────────────────
# FINAL SUMMARY
# ─────────────────────────────────────────────
best_params = study.best_params
print("\n Best hyperparameters found:")
for k, v in best_params.items():
    print(f" {k}: {v}")

# Append study-level summary to JSON
summary = {
    "best_trial_number": study.best_trial.number,
    "best_val_auc": study.best_value,
    "best_hyperparameters": best_params,
    "n_trials_completed": len(study.trials),
}
with open(str(RESULTS_DIR / "saint_lc_trials.json"), "r") as f:
    all_data = json.load(f)

all_data.append({"study_summary": summary})
with open(str(RESULTS_DIR / "saint_lc_trials.json"), "w") as f:
    json.dump(all_data, f, indent=4)

print(f"\n All results saved to saint_optuna_trials.json")
print(f" Best Val AUC: {study.best_value:.4f}")